# Compare SAM3-seg — Fine-Tuning Analysis

Structured analysis of SAM3 fine-tuning on ISIC 2018 Task 1, designed to
mirror the YOLO26 `compare_models` notebook for **direct cross-model
comparison**.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path
from typing import List, Dict, Tuple

import json
import cv2
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from matplotlib.patches import Patch
from pycocotools import mask as mask_utils

warnings.filterwarnings("ignore", category=FutureWarning)

# ----- global style (same as YOLO26 notebook) ---------------------------------
plt.style.use("seaborn-v0_8-whitegrid")
mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "legend.fontsize": 10,
})
sns.set_context("notebook", font_scale=1.0)

# ----- colour palette (Okabe-Ito, consistent with YOLO26 notebook) -----------
METRIC_PALETTE = {
    "Segm AP":    "#0072B2",
    "Segm AP@50": "#56B4E9",
    "Segm AP@75": "#009E73",
    "BBox AP":    "#E69F00",
    "BBox AP@50": "#D55E00",
    "BBox AP@75": "#CC79A7",
}

LOSS_PALETTE = {
    "Total Loss": "#222222",
    "BBox Loss":  "#E63946",
    "GIoU Loss":  "#2A9D8F",
    "CE Loss":    "#0072B2",
    "Mask Loss":  "#E69F00",
    "Dice Loss":  "#D55E00",
}

SUCCESS = "#2A9D8F"
DANGER  = "#E63946"
NEUTRAL = "#6C757D"

print("Setup OK")

### 1.1 Path Configuration

Adjust `unb_server` and `BASE_DIR` to match your environment.
The notebook auto-detects whether each file exists and skips
gracefully.

In [ ]:
unb_server = True

if unb_server:
    BASE_DIR = Path("/home/antoniovinicius/projects/sandbox_sam3")
else:
    BASE_DIR = Path("/home/avmoura_linux/Documents/unb/sandbox_sam3")

LOG_DIR = BASE_DIR / "logs" / "sam3_ft_isic_2018_task_1_improved_3"

DATASET_DIR = BASE_DIR / "datasets" / "isic_2018_task1_coco"

CONFIG = {
    "train_stats":  LOG_DIR / "logs" / "train_stats.json",
    "val_stats":    LOG_DIR / "logs" / "val_stats.json",
    "best_stats":   LOG_DIR / "logs" / "best_stats.json",
    "config_yaml":  LOG_DIR / "config.yaml",
    "pred_json":    LOG_DIR / "dumps" / "coco_predictions_segm.json",
    "images_dir":   DATASET_DIR / "valid",
    "gt_json":      DATASET_DIR / "valid" / "_annotations.coco.json",
    "test_images_dir": DATASET_DIR / "test",
    "test_gt_json":    DATASET_DIR / "test" / "_annotations.coco.json",
    "model_weights":   LOG_DIR / "checkpoints" / "checkpoint.pt",
    "score_threshold": 0.35,
    "mask_alpha": 0.45,
}

# Anchor ISIC IDs for qualitative comparison --- same lesion images used in
# the YOLO26 compare_models notebook so the two can be placed side-by-side.
# We match by ISIC ID prefix because the roboflow hashes differ between the
# YOLO26 and SAM3 dataset exports.
ANCHOR_ISIC_IDS = [
    "ISIC_0017389",
    "ISIC_0015169",
    "ISIC_0015970",
    "ISIC_0017434",
    "ISIC_0020081",
    "ISIC_0036343",
]

# Sanity check
print(f"LOG_DIR:   {LOG_DIR}  {'[OK]' if LOG_DIR.exists() else '[MISSING]'}") 
print(f"DATASET:   {DATASET_DIR}  {'[OK]' if DATASET_DIR.exists() else '[MISSING]'}") 
for name, p in CONFIG.items():
    if isinstance(p, Path):
        status = "[OK]" if p.exists() else "[MISSING]"
        print(f"  {name:<20s} {status}  {p}")

## 2. Helpers

In [ ]:
def load_jsonl(filepath: Path) -> pd.DataFrame:
    """Load a JSONL (one-JSON-per-line) file into a DataFrame."""
    if not filepath.exists():
        print(f"[SKIP] File not found: {filepath}")
        return pd.DataFrame()
    data = []
    with open(filepath, "r") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))
    return pd.DataFrame(data)


def smooth(x: np.ndarray, window: int = 5) -> np.ndarray:
    """Centered moving average (NaN-safe)."""
    return pd.Series(x).rolling(window=window, center=True, min_periods=1).mean().to_numpy()


def epochs_to_threshold(series: pd.Series, frac: float = 0.95) -> int:
    """First epoch where the metric reaches `frac` of its overall best."""
    if series.dropna().empty:
        return np.nan
    target = series.max() * frac
    hit = series[series >= target]
    return int(hit.index[0]) if not hit.empty else np.nan


def stability_std(series: pd.Series, last_n: int = 10) -> float:
    """Std of the last `last_n` values --- lower = more stable."""
    tail = series.dropna().tail(last_n)
    return float(tail.std()) if not tail.empty else np.nan


print("Helpers ready")

## 3. Data Loading

In [ ]:
df_train = load_jsonl(CONFIG["train_stats"])
df_val   = load_jsonl(CONFIG["val_stats"])
df_best  = load_jsonl(CONFIG["best_stats"])

print(f"Training epochs loaded: {len(df_train)}")
print(f"Validation epochs loaded: {len(df_val)}")

# ---- Normalise epoch column --------------------------------------------------
if "Trainer/epoch" in df_train.columns:
    df_train["epoch"] = df_train["Trainer/epoch"].astype(int)
if "Trainer/epoch" in df_val.columns:
    df_val["epoch"] = df_val["Trainer/epoch"].astype(int)

# ---- Shorten metric names for convenience ------------------------------------
VAL_METRIC_MAP = {
    "Meters_train/val_isic_val/detection/coco_eval_segm_AP":     "Segm AP",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AP_50":  "Segm AP@50",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AP_75":  "Segm AP@75",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AP_large":  "Segm AP (large)",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AP_medium": "Segm AP (medium)",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AR_maxDets@1":   "Segm AR@1",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AR_maxDets@10":  "Segm AR@10",
    "Meters_train/val_isic_val/detection/coco_eval_segm_AR_maxDets@100": "Segm AR@100",
    "Meters_train/val_isic_val/detection/coco_eval_bbox_AP":     "BBox AP",
    "Meters_train/val_isic_val/detection/coco_eval_bbox_AP_50":  "BBox AP@50",
    "Meters_train/val_isic_val/detection/coco_eval_bbox_AP_75":  "BBox AP@75",
    "Meters_train/val_isic_val/detection/coco_eval_bbox_AR_maxDets@100": "BBox AR@100",
}

for long_name, short_name in VAL_METRIC_MAP.items():
    if long_name in df_val.columns:
        df_val[short_name] = df_val[long_name]

# ---- Training loss short names -----------------------------------------------
TRAIN_LOSS_MAP = {
    "Losses/train_all_loss":      "Total Loss",
    "Losses/train_all_loss_bbox": "BBox Loss",
    "Losses/train_all_loss_giou": "GIoU Loss",
    "Losses/train_all_loss_ce":   "CE Loss",
    "Losses/train_all_loss_mask": "Mask Loss",
    "Losses/train_all_loss_dice": "Dice Loss",
}
for long_name, short_name in TRAIN_LOSS_MAP.items():
    if long_name in df_train.columns:
        df_train[short_name] = df_train[long_name]

display(Markdown("**Validation columns available:**"))
print([c for c in df_val.columns if not c.startswith("Meters_") and not c.startswith("Losses/") and not c.startswith("Trainer/")])

## 4. Executive Summary

In [ ]:
if "Segm AP" in df_val.columns:
    best_idx = df_val["Segm AP"].idxmax()
    best_row = df_val.loc[best_idx]
    best_ep = int(best_row["epoch"])
    best_ap = best_row["Segm AP"]

    display(Markdown(
        f"### Best checkpoint: **epoch {best_ep}** "
        f"(Segm AP = **{best_ap:.4f}**)"
    ))

    # Summary table
    summary_metrics = [
        "Segm AP", "Segm AP@50", "Segm AP@75",
        "BBox AP", "BBox AP@50", "BBox AP@75",
        "Segm AR@100", "BBox AR@100",
    ]
    summary = {}
    for m in summary_metrics:
        if m in df_val.columns:
            summary[m] = {
                "Best (any epoch)": df_val[m].max(),
                "@ best Segm AP ep": best_row.get(m, np.nan),
                "Last epoch":        df_val[m].iloc[-1],
            }
    df_summary = pd.DataFrame(summary).T

    display(Markdown("### 4.1 Key metrics at best checkpoint vs last epoch"))
    display(
        df_summary.style
        .format("{:.4f}", na_rep="\u2014")
        .background_gradient(subset=["Best (any epoch)"], cmap="YlGn")
    )

    # Convergence & stability
    segm_ap_series = df_val.set_index("epoch")["Segm AP"]
    ep95 = epochs_to_threshold(segm_ap_series, 0.95)
    ep99 = epochs_to_threshold(segm_ap_series, 0.99)
    stab = stability_std(segm_ap_series, last_n=10)

    display(Markdown(
        f"### 4.2 Convergence & stability\n"
        f"| Metric | Value |\n|---|---|\n"
        f"| Epochs to 95% of best | **{ep95}** |\n"
        f"| Epochs to 99% of best | **{ep99}** |\n"
        f"| Stability (std last 10 ep) | **{stab:.4f}** |\n"
        f"| Total epochs | **{int(df_val['epoch'].max())}** |"
    ))
else:
    print("[SKIP] Segm AP not found in validation data.")

## 5. Training Loss Curves

In [ ]:
WINDOW = 3

loss_cols = {
    "Total Loss": LOSS_PALETTE["Total Loss"],
    "BBox Loss":  LOSS_PALETTE["BBox Loss"],
    "GIoU Loss":  LOSS_PALETTE["GIoU Loss"],
    "CE Loss":    LOSS_PALETTE["CE Loss"],
    "Mask Loss":  LOSS_PALETTE["Mask Loss"],
    "Dice Loss":  LOSS_PALETTE["Dice Loss"],
}

available_losses = {k: v for k, v in loss_cols.items() if k in df_train.columns}
n_plots = len(available_losses)

if n_plots > 0:
    fig, axes = plt.subplots(n_plots, 1, figsize=(12, 3.5 * n_plots), squeeze=False)
    for i, (name, color) in enumerate(available_losses.items()):
        ax = axes[i, 0]
        y = df_train[name].to_numpy()
        eps = df_train["epoch"].to_numpy()
        ax.plot(eps, y, color=color, alpha=0.25, lw=1)
        ax.plot(eps, smooth(y, WINDOW), color=color, lw=2.2)
        ax.set_title(f"Training {name}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] No training loss columns found.")

## 6. Validation Metrics — Learning Curves

In [ ]:
WINDOW = 3

# (a) Segm AP family
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

segm_metrics = ["Segm AP", "Segm AP@50", "Segm AP@75"]
for m in segm_metrics:
    if m in df_val.columns:
        y = df_val[m].to_numpy()
        eps = df_val["epoch"].to_numpy()
        color = METRIC_PALETTE.get(m, "#888")
        axes[0].plot(eps, y, color=color, alpha=0.2, lw=1)
        axes[0].plot(eps, smooth(y, WINDOW), color=color, lw=2.2, label=m)
        b = int(np.argmax(y))
        axes[0].scatter([eps[b]], [y[b]], color=color, s=50, zorder=5,
                        edgecolor="white", lw=1)
axes[0].set_title("Segmentation AP")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("AP")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

# (b) BBox AP family
bbox_metrics = ["BBox AP", "BBox AP@50", "BBox AP@75"]
for m in bbox_metrics:
    if m in df_val.columns:
        y = df_val[m].to_numpy()
        eps = df_val["epoch"].to_numpy()
        color = METRIC_PALETTE.get(m, "#888")
        axes[1].plot(eps, y, color=color, alpha=0.2, lw=1)
        axes[1].plot(eps, smooth(y, WINDOW), color=color, lw=2.2, label=m)
        b = int(np.argmax(y))
        axes[1].scatter([eps[b]], [y[b]], color=color, s=50, zorder=5,
                        edgecolor="white", lw=1)
axes[1].set_title("BBox AP")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AP")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

fig.suptitle("Validation AP over training --- SAM3 fine-tuned (improved_3)",
             y=1.03, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Learning Curve — Training Loss vs Validation AP

In [ ]:
if "Total Loss" in df_train.columns and "Segm AP" in df_val.columns:
    fig, ax1 = plt.subplots(figsize=(12, 5.5))

    eps_t = df_train["epoch"].to_numpy()
    eps_v = df_val["epoch"].to_numpy()

    # Left axis: Training total loss
    color1 = LOSS_PALETTE["Total Loss"]
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Training Total Loss", color=color1)
    ax1.plot(eps_t, smooth(df_train["Total Loss"].to_numpy(), 3),
             color=color1, lw=2, label="Train Loss")
    ax1.tick_params(axis="y", labelcolor=color1)
    ax1.grid(True, alpha=0.3)

    # Right axis: Validation Segm AP
    ax2 = ax1.twinx()
    color2 = METRIC_PALETTE["Segm AP"]
    ax2.set_ylabel("Validation Segm AP", color=color2)
    ax2.plot(eps_v, smooth(df_val["Segm AP"].to_numpy(), 3),
             color=color2, lw=2, marker="o", markersize=4, label="Val Segm AP")
    ax2.tick_params(axis="y", labelcolor=color2)

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    plt.title("Training Loss vs Validation AP --- SAM3 (improved_3)",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] Missing training loss or validation AP data.")

## 8. Convergence Analysis

In [ ]:
if "Segm AP" in df_val.columns:
    segm_ap = df_val.set_index("epoch")["Segm AP"]
    best_val = segm_ap.max()
    best_ep  = segm_ap.idxmax()

    thresholds = [0.90, 0.95, 0.99]
    conv_data = []
    for t in thresholds:
        ep = epochs_to_threshold(segm_ap, t)
        conv_data.append({
            "Threshold": f"{t:.0%} of best",
            "Target value": f"{best_val * t:.4f}",
            "Reached at epoch": ep,
        })
    df_conv = pd.DataFrame(conv_data)
    display(df_conv.style.hide(axis="index"))

    # Convergence plot
    fig, ax = plt.subplots(figsize=(10, 4.5))
    eps = segm_ap.index.to_numpy()
    y = segm_ap.to_numpy()
    ax.plot(eps, y, color=METRIC_PALETTE["Segm AP"], alpha=0.25, lw=1)
    ax.plot(eps, smooth(y, 3), color=METRIC_PALETTE["Segm AP"], lw=2.5)

    for t, ls in zip(thresholds, [":", "--", "-."]):
        val = best_val * t
        ax.axhline(val, color=NEUTRAL, ls=ls, lw=1, alpha=0.6)
        ax.annotate(f"{t:.0%}  ({val:.4f})", xy=(eps[-1], val),
                    xytext=(5, 0), textcoords="offset points",
                    fontsize=8, color=NEUTRAL, va="center")

    ax.scatter([best_ep], [best_val], s=100, color=SUCCESS,
               edgecolor="white", lw=1.5, zorder=5, label=f"Best ep {best_ep}")
    ax.set_title("Convergence --- Segm AP with threshold lines")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Segm AP")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] Segm AP not found.")

## 9. Training Stability

In [ ]:
stability_metrics = ["Segm AP", "Segm AP@50", "BBox AP"]
stab_data = []
for m in stability_metrics:
    if m in df_val.columns:
        s = df_val[m].dropna()
        stab_data.append({
            "Metric": m,
            "Std (last 10 ep)": stability_std(s, 10),
            "Std (last 5 ep)":  stability_std(s, 5),
            "Range (max-min)":  s.max() - s.min(),
            "CoV (%)":          100 * s.std() / s.mean() if s.mean() > 0 else np.nan,
        })

if stab_data:
    df_stab = pd.DataFrame(stab_data).set_index("Metric")
    display(df_stab.style.format("{:.4f}", na_rep="\u2014")
            .background_gradient(cmap="RdYlGn_r"))

    # Rolling std plot
    fig, ax = plt.subplots(figsize=(10, 4))
    for m in stability_metrics:
        if m in df_val.columns:
            rolling_std = df_val[m].rolling(5, center=True, min_periods=2).std()
            ax.plot(df_val["epoch"], rolling_std,
                    color=METRIC_PALETTE.get(m, "#888"), lw=2, label=m)
    ax.set_title("Rolling Std (window=5) --- lower is more stable")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Rolling Std")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] No stability metrics available.")

## 10. Full Metrics Table — All Epochs

In [ ]:
display_cols = [c for c in [
    "epoch",
    "Segm AP", "Segm AP@50", "Segm AP@75",
    "BBox AP", "BBox AP@50", "BBox AP@75",
    "Segm AR@100", "BBox AR@100",
    "Segm AP (large)", "Segm AP (medium)",
] if c in df_val.columns]

if display_cols:
    df_display = df_val[display_cols].copy()

    def highlight_max(s):
        is_max = s == s.max()
        return [
            f"background-color: {SUCCESS}; color: white; font-weight: 600" if v else ""
            for v in is_max
        ]

    display(
        df_display.style
        .apply(highlight_max, axis=0,
               subset=[c for c in display_cols if c != "epoch"])
        .format("{:.4f}", subset=[c for c in display_cols if c != "epoch"],
                na_rep="\u2014")
        .format("{:.0f}", subset=["epoch"])
        .set_caption("Green = best value for each metric across epochs")
    )
else:
    print("[SKIP] No display columns available.")

## 11. Geometric Robustness — Retention Ratio

Retention = AP / AP@50.  Higher retention means the model produces
tighter, more accurate masks (not just correct at IoU 0.50).

In [ ]:
if "Segm AP" in df_val.columns and "Segm AP@50" in df_val.columns:
    df_val["Segm retention"] = df_val["Segm AP"] / df_val["Segm AP@50"]

if "BBox AP" in df_val.columns and "BBox AP@50" in df_val.columns:
    df_val["BBox retention"] = df_val["BBox AP"] / df_val["BBox AP@50"]

ret_cols = [c for c in ["Segm retention", "BBox retention"] if c in df_val.columns]
if ret_cols:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for c in ret_cols:
        color = METRIC_PALETTE.get(c.replace(" retention", " AP"), "#888")
        y = df_val[c].to_numpy()
        ax.plot(df_val["epoch"], y, color=color, alpha=0.2, lw=1)
        ax.plot(df_val["epoch"], smooth(y, 3), color=color, lw=2.2, label=c)
    ax.set_title("Retention = AP / AP@50 --- geometric robustness over training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Retention ratio")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    last = df_val.iloc[-1]
    display(Markdown(
        f"**Last epoch retention:** "
        f"Segm = {last.get('Segm retention', np.nan):.4f}, "
        f"BBox = {last.get('BBox retention', np.nan):.4f}"
    ))
else:
    print("[SKIP] Retention columns not available.")

## 12. Recall Analysis (AR)

In [ ]:
ar_cols = [c for c in ["Segm AR@1", "Segm AR@10", "Segm AR@100"] if c in df_val.columns]
if ar_cols:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    palette = ["#0072B2", "#009E73", "#E69F00"]
    for c, color in zip(ar_cols, palette):
        y = df_val[c].to_numpy()
        ax.plot(df_val["epoch"], y, color=color, alpha=0.2, lw=1)
        ax.plot(df_val["epoch"], smooth(y, 3), color=color, lw=2.2, label=c)
    ax.set_title("Segmentation Average Recall (AR) over training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("AR")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] AR columns not available.")

## 13. Qualitative Results — Validation Set

Overlay of Ground Truth (green) and SAM3 predictions (red) on validation
images, using the `coco_predictions_segm.json` dump.

In [ ]:
import random

NUM_IMAGES_TO_SHOW = 10
OVERLAY_MASKS = True

def load_json_data(pred_path: Path, gt_path: Path):
    preds_map, gt_map, id_to_filename = {}, {}, {}

    if pred_path.exists():
        with open(pred_path, "r") as f:
            preds = json.load(f)
        for p in preds:
            preds_map.setdefault(p["image_id"], []).append(p)

    if gt_path.exists():
        with open(gt_path, "r") as f:
            gt = json.load(f)
        for img in gt["images"]:
            id_to_filename[img["id"]] = img["file_name"]
        for ann in gt.get("annotations", []):
            gt_map.setdefault(ann["image_id"], []).append(ann)

    return preds_map, gt_map, id_to_filename


def apply_mask(image, mask, color, alpha):
    for c in range(3):
        image[:, :, c] = np.where(
            mask == 1,
            image[:, :, c] * (1 - alpha) + alpha * color[c],
            image[:, :, c],
        )
    return image


def draw_visualizations(img, annotations, threshold=0.0,
                        is_gt=False, color_override=None):
    count = 0
    img_h, img_w = img.shape[:2]
    if not is_gt:
        annotations.sort(key=lambda x: x.get("score", 0))

    for ann in annotations:
        score = ann.get("score", 1.0)
        if not is_gt and score < threshold:
            continue

        color = color_override or ([0, 255, 0] if is_gt else [random.randint(50, 255) for _ in range(3)])

        if "segmentation" in ann:
            try:
                segm = ann["segmentation"]
                binary_mask = np.zeros((img_h, img_w), dtype=np.uint8)
                if isinstance(segm, list):
                    for poly in segm:
                        pts = np.array(poly).reshape(-1, 2).astype(np.int32)
                        cv2.fillPoly(binary_mask, [pts], 1)
                elif isinstance(segm, dict):
                    binary_mask = mask_utils.decode(segm)
                    if binary_mask.shape[:2] != (img_h, img_w):
                        binary_mask = cv2.resize(binary_mask, (img_w, img_h),
                                                 interpolation=cv2.INTER_NEAREST)
                img = apply_mask(img, binary_mask, color, CONFIG["mask_alpha"])
                contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL,
                                               cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(img, contours, -1, (255, 255, 255), 1)
            except Exception:
                pass

        if "bbox" in ann:
            bbox = ann["bbox"]
            if all(v <= 1.1 for v in bbox):
                x, y_, w, h = int(bbox[0]*img_w), int(bbox[1]*img_h), int(bbox[2]*img_w), int(bbox[3]*img_h)
            else:
                x, y_, w, h = map(int, bbox)
            cv2.rectangle(img, (x, y_), (x + w, y_ + h), color, 2)
            if not is_gt:
                label = f"Pred: {score:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(img, (x, y_ - 20), (x + tw, y_), color, -1)
                cv2.putText(img, label, (x, y_ - 5), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (255, 255, 255), 1)
        count += 1

    return count, img


# ---- Run visualisation -------------------------------------------------------
if CONFIG["pred_json"].exists() and CONFIG["gt_json"].exists():
    preds_map, gt_map, id_to_filename = load_json_data(
        CONFIG["pred_json"], CONFIG["gt_json"]
    )
    shown = 0
    for img_id, predictions in preds_map.items():
        if shown >= NUM_IMAGES_TO_SHOW:
            break
        if img_id not in id_to_filename:
            continue
        img_path = CONFIG["images_dir"] / id_to_filename[img_id]
        if not img_path.exists():
            continue
        original_img = cv2.imread(str(img_path))
        if original_img is None:
            continue

        gt_annotations = gt_map.get(img_id, [])
        combined = original_img.copy()
        _, combined = draw_visualizations(combined, gt_annotations, is_gt=True,
                                          color_override=[113, 204, 46])
        _, combined = draw_visualizations(combined, predictions,
                                          CONFIG["score_threshold"],
                                          is_gt=False, color_override=[60, 20, 220])

        combined_rgb = cv2.cvtColor(combined, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 10))
        plt.imshow(combined_rgb)
        plt.title(f"Image: {id_to_filename[img_id]}\nGT (Green) | Prediction (Red)",
                  fontsize=14)
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        shown += 1

    if shown == 0:
        print("[SKIP] No images could be loaded (dataset may not be available locally).")
else:
    print("[SKIP] Prediction JSON or GT JSON not found ---\n"
          "       run this on a machine with the ISIC 2018 dataset.")

## 14. Model Inference — Test Set

Runs SAM3 inference on the **test split** using the fine-tuned
checkpoint.  The same `IMAGE_STEMS` used in the YOLO26
`compare_models` notebook are targeted first to allow **direct
side-by-side comparison** between the two models.

> **Requires**: SAM3 package installed, GPU, and the ISIC 2018 dataset.

In [ ]:
import time
import torch

try:
    from PIL import Image
    from sam3 import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor
    _sam3_available = True
except ImportError as e:
    print(f"[SKIP] SAM3 not available: {e}")
    _sam3_available = False

if _sam3_available:
    src_ckpt   = LOG_DIR / "checkpoints" / "checkpoint.pt"
    fixed_ckpt = LOG_DIR / "checkpoints" / "checkpoint_for_inference.pt"

    if src_ckpt.exists() and not fixed_ckpt.exists():
        print(f"Re-prefixing {src_ckpt} -> {fixed_ckpt}")
        raw = torch.load(src_ckpt, map_location="cpu", weights_only=True)
        state = raw["model"] if "model" in raw and isinstance(raw["model"], dict) else raw
        fixed = {"model": {f"detector.{k}": v for k, v in state.items()}}
        torch.save(fixed, fixed_ckpt)
        print(f"OK. {len(state)} keys re-prefixed.")

    ckpt_path = fixed_ckpt if fixed_ckpt.exists() else src_ckpt
    if ckpt_path.exists():
        INFERENCE_CONFIG = {
            "model_weights": ckpt_path,
            "prompt": "skin cancer",
            "conf_threshold": 0.27,
            "mask_alpha": 0.45,
        }
        print(f"Loading SAM3 model from: {ckpt_path}")
        model = build_sam3_image_model(checkpoint_path=str(ckpt_path))
        processor = Sam3Processor(model, confidence_threshold=INFERENCE_CONFIG["conf_threshold"])
        print("Model loaded and ready for inference!")
    else:
        print(f"[SKIP] Checkpoint not found at {src_ckpt}")
        _sam3_available = False

In [ ]:
def draw_sam3_predictions(img_cv, results, alpha=0.45, color_override=None):
    result_img = img_cv.copy()
    total_objects = len(results.get("scores", []))
    h, w = result_img.shape[:2]

    for i in range(total_objects):
        color = color_override or [random.randint(50, 255) for _ in range(3)]
        prob = results["scores"][i].item()

        if "masks" in results and len(results["masks"]) > i:
            mask = results["masks"][i].squeeze(0).cpu().numpy()
            if mask.shape != (h, w):
                if mask.shape == (w, h):
                    mask = mask.T
                else:
                    mask = cv2.resize(mask.astype(np.uint8), (w, h),
                                      interpolation=cv2.INTER_NEAREST).astype(bool)
            mask_bool = mask.astype(bool)
            overlay = result_img.copy()
            overlay[mask_bool] = color
            result_img = cv2.addWeighted(result_img, 1 - alpha, overlay, alpha, 0)
            contours, _ = cv2.findContours(mask_bool.astype(np.uint8) * 255,
                                           cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(result_img, contours, -1, color, max(1, 2 // 2))

        if "boxes" in results and len(results["boxes"]) > i:
            box = results["boxes"][i].cpu().numpy().astype(int)
            x1, y1, x2, y2 = box
            cv2.rectangle(result_img, (x1, y1), (x2, y2), color, 2)
            label = f"Pred: {prob:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            label_y = max(y1, th + 3)
            cv2.rectangle(result_img, (x1, label_y - th - 3),
                          (x1 + tw + 6, label_y), color, -1)
            cv2.putText(result_img, label, (x1 + 3, label_y - 3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return result_img


def _resolve_image_by_isic_id(isic_id: str, search_dirs: list) -> Path:
    """Find an image file whose name starts with `isic_id` in any of the
    given directories.  Returns the first match or None."""
    for d in search_dirs:
        if not d.exists():
            continue
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            for p in d.glob(ext):
                if p.name.startswith(isic_id):
                    return p
    return None


def _iou_pixel(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter) / float(union) if union > 0 else float("nan")


# ---- Run inference on ANCHOR_ISIC_IDS (YOLO26-consistent) + additional test --
COLOR_GT_BGR   = [113, 204, 46]    # lime green
COLOR_PRED_BGR = [60, 20, 220]     # crimson red

# Directories to search for anchor images (test first, then valid, then train)
_search_dirs = [
    CONFIG["test_images_dir"],
    CONFIG["images_dir"],                    # valid
    DATASET_DIR / "train",
]

if _sam3_available:
    # --- Load GT annotations from whichever splits are available --------------
    all_gt_map = {}
    all_id_to_filename = {}
    all_filename_to_dir = {}

    for split_name, img_dir, gt_json in [
        ("test",  CONFIG["test_images_dir"], CONFIG["test_gt_json"]),
        ("valid", CONFIG["images_dir"],      CONFIG["gt_json"]),
        ("train", DATASET_DIR / "train",     DATASET_DIR / "train" / "_annotations.coco.json"),
    ]:
        if gt_json.exists():
            _, gt_map_tmp, id2fn_tmp = load_json_data(Path("dummy"), gt_json)
            for img_id, anns in gt_map_tmp.items():
                all_gt_map[f"{split_name}_{img_id}"] = anns
            for img_id, fn in id2fn_tmp.items():
                all_id_to_filename[f"{split_name}_{img_id}"] = fn
                all_filename_to_dir[fn] = img_dir

    # --- Resolve anchor images by ISIC ID prefix -----------------------------
    anchor_files = []
    anchor_isic_matched = []
    for isic_id in ANCHOR_ISIC_IDS:
        p = _resolve_image_by_isic_id(isic_id, _search_dirs)
        if p is not None:
            anchor_files.append(p)
            anchor_isic_matched.append(isic_id)
            print(f"  [ANCHOR] {isic_id} -> {p}")
        else:
            print(f"  [MISS]   {isic_id} not found in any split")

    # --- Build queue: anchors first, then remaining test images ---------------
    test_files_all = sorted(CONFIG["test_images_dir"].glob("*.jpg")) if CONFIG["test_images_dir"].exists() else []
    remaining = [f for f in test_files_all if f not in anchor_files]
    test_queue = anchor_files + remaining

    max_show = max(len(ANCHOR_ISIC_IDS), 10)
    shown = 0
    inference_times = []

    print(f"\nRunning inference on up to {max_show} images "
          f"({len(anchor_files)} anchor + remaining)...\n")

    for img_path in test_queue:
        if shown >= max_show:
            break

        original_bgr = cv2.imread(str(img_path))
        if original_bgr is None:
            continue

        # Find GT annotations for this image
        gt_annotations = []
        for key, fn in all_id_to_filename.items():
            if fn == img_path.name and all_filename_to_dir.get(fn) == img_path.parent:
                gt_annotations = all_gt_map.get(key, [])
                break

        image_pil = Image.open(img_path)
        t0 = time.perf_counter()
        state = processor.set_image(image_pil)
        results = processor.set_text_prompt(state=state,
                                            prompt=INFERENCE_CONFIG["prompt"])
        dt = time.perf_counter() - t0
        inference_times.append(dt)

        combined = original_bgr.copy()
        _, combined = draw_visualizations(combined, gt_annotations, is_gt=True,
                                          color_override=COLOR_GT_BGR)
        combined = draw_sam3_predictions(combined, results,
                                         alpha=INFERENCE_CONFIG["mask_alpha"],
                                         color_override=COLOR_PRED_BGR)

        is_anchor = any(img_path.name.startswith(aid) for aid in ANCHOR_ISIC_IDS)
        tag = " [ANCHOR]" if is_anchor else ""

        combined_rgb = cv2.cvtColor(combined, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 10))
        plt.imshow(combined_rgb)
        plt.title(f"Image: {img_path.name}{tag}\n"
                  f"GT (Green) | SAM3 Prediction (Red)  ---  {dt*1000:.0f} ms",
                  fontsize=13)
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        shown += 1

    if inference_times:
        avg_ms = np.mean(inference_times) * 1000
        print(f"\n--- INFERENCE SUMMARY ---")
        print(f"Images processed: {len(inference_times)}")
        print(f"Avg inference time: {avg_ms:.1f} ms/image")
    if not anchor_files:
        print("\n[WARNING] No anchor images found. Make sure the ISIC 2018 "
              "dataset is available at DATASET_DIR.")
elif not _sam3_available:
    print("[SKIP] SAM3 not available --- install sam3 package and ensure GPU access.")
else:
    print(f"[SKIP] Test images dir not found: {CONFIG['test_images_dir']}")

## 15. Metrics Presentation — Final Epoch

In [ ]:
if not df_val.empty:
    last = df_val.iloc[-1]
    epoch_num = int(last.get("epoch", -1))

    print("=" * 60)
    print(f"  SAM3 VALIDATION RESULTS (Epoch {epoch_num})")
    print("=" * 60)

    segm_cols = [c for c in df_val.columns
                 if c.startswith("Segm") and "retention" not in c]
    bbox_cols = [c for c in df_val.columns
                 if c.startswith("BBox") and "retention" not in c]

    print("\n  --- Segmentation (Mask) ---")
    for c in sorted(segm_cols):
        v = last[c]
        if pd.notna(v) and v != -1.0:
            print(f"    {c:<25s} {v:.4f}")

    print("\n  --- Bounding Box ---")
    for c in sorted(bbox_cols):
        v = last[c]
        if pd.notna(v) and v != -1.0:
            print(f"    {c:<25s} {v:.4f}")

    print("=" * 60)
else:
    print("[SKIP] No validation data.")

## 16. Automatic Insights

In [ ]:
insights = []

if "Segm AP" in df_val.columns:
    best_idx = df_val["Segm AP"].idxmax()
    best = df_val.loc[best_idx]
    insights.append(
        f"**Best Segm AP**: **{best['Segm AP']:.4f}** at epoch "
        f"**{int(best['epoch'])}** (AP@50 = {best.get('Segm AP@50', np.nan):.4f}, "
        f"AP@75 = {best.get('Segm AP@75', np.nan):.4f})."
    )

    last = df_val.iloc[-1]
    delta = last["Segm AP"] - best["Segm AP"]
    if abs(delta) > 0.001:
        direction = "dropped" if delta < 0 else "improved"
        insights.append(
            f"**Last epoch vs best**: Segm AP {direction} by "
            f"**{abs(delta):.4f}** (last = {last['Segm AP']:.4f})."
        )
    else:
        insights.append("**Last epoch = best epoch** (no AP drop).")

    segm_ap_series = df_val.set_index("epoch")["Segm AP"]
    stab = stability_std(segm_ap_series, 10)
    if stab < 0.005:
        insights.append(
            f"**High stability**: std over last 10 epochs = {stab:.4f} "
            f"(very consistent training)."
        )
    elif stab > 0.01:
        insights.append(
            f"**Moderate instability**: std over last 10 epochs = {stab:.4f} "
            f"(consider longer training or lower LR)."
        )

if "Segm retention" in df_val.columns:
    ret = df_val["Segm retention"].iloc[-1]
    if ret > 0.76:
        insights.append(
            f"**Strong geometric robustness**: Segm retention = {ret:.4f} "
            f"(masks are tight at higher IoU thresholds)."
        )
    else:
        insights.append(
            f"**Moderate geometric robustness**: Segm retention = {ret:.4f} "
            f"(room for improvement at strict IoU)."
        )

display(Markdown("### Insights"))
for s in insights:
    display(Markdown(f"- {s}"))

## 17. Cross-Model Reference — SAM3 vs YOLO26

Manually paste or load the YOLO26 champion metrics here for
direct comparison.  The table below pre-fills SAM3 best-epoch
values and leaves a placeholder for YOLO26.

In [ ]:
# ---- Pre-fill SAM3 metrics --------------------------------------------------
sam3_metrics = {}
if "Segm AP" in df_val.columns:
    best_idx = df_val["Segm AP"].idxmax()
    best = df_val.loc[best_idx]
    sam3_metrics = {
        "Segm AP":    best.get("Segm AP", np.nan),
        "Segm AP@50": best.get("Segm AP@50", np.nan),
        "Segm AP@75": best.get("Segm AP@75", np.nan),
        "BBox AP":    best.get("BBox AP", np.nan),
        "BBox AP@50": best.get("BBox AP@50", np.nan),
        "Segm AR@100": best.get("Segm AR@100", np.nan),
    }

# ---- YOLO26 champion (update from compare_models notebook) ------------------
# Fill these values from the YOLO26 compare_models notebook output.
yolo26_metrics = {
    "Segm AP":    np.nan,  # Mask mAP@50-95
    "Segm AP@50": np.nan,  # Mask mAP@50
    "Segm AP@75": np.nan,
    "BBox AP":    np.nan,  # BBox mAP@50-95
    "BBox AP@50": np.nan,  # BBox mAP@50
    "Segm AR@100": np.nan,
}

df_compare = pd.DataFrame({
    "SAM3 (improved_3)": sam3_metrics,
    "YOLO26 (champion)": yolo26_metrics,
})
df_compare["Delta (SAM3 - YOLO26)"] = df_compare["SAM3 (improved_3)"] - df_compare["YOLO26 (champion)"]

display(Markdown("### Direct comparison (fill in YOLO26 values from that notebook)"))
display(df_compare.style.format("{:.4f}", na_rep="\u2014"))

## 18. Export — CSV Summary

In [ ]:
export_cols = [c for c in [
    "epoch",
    "Segm AP", "Segm AP@50", "Segm AP@75",
    "Segm AP (large)", "Segm AP (medium)",
    "Segm AR@1", "Segm AR@10", "Segm AR@100",
    "BBox AP", "BBox AP@50", "BBox AP@75", "BBox AR@100",
    "Segm retention", "BBox retention",
] if c in df_val.columns]

if export_cols:
    df_export = df_val[export_cols].copy()
    out_csv = LOG_DIR / "sam3_compare_models_summary.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df_export.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}  ({len(df_export)} rows x {df_export.shape[1]} cols)")
    display(df_export.tail(5))
else:
    print("[SKIP] No columns to export.")